# 02 - Clean + Filter + Impute (ALI dataset)

This notebook:
1) Loads `data/raw/wdi_2022_initial.csv`
2) Removes aggregates using the World Bank country metadata (region != "Aggregates")
3) Selects the baseline indicator set (dropping PM2.5 and homicide due to missingness)
4) Applies missingness rules (drop countries with too much missing data)
5) Imputes remaining missing values (KNN imputation baseline)
6) Saves `data/processed/ali_2022_imputed.csv`

In [ ]:
import pandas as pd
import numpy as np
import requests

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import KNNImputer

In [ ]:
RAW_PATH = "../data/raw/wdi_2022_initial.csv"
df = pd.read_csv(RAW_PATH)
df.head(), df.shape

In [ ]:
indicator_cols_all = [
    "gdp_per_capita_ppp_const",
    "unemployment_rate",
    "labor_force_participation",
    "life_expectancy",
    "pm25_exposure",
    "homicide_rate",
    "ppp_conversion_factor"
]

for c in indicator_cols_all:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df[indicator_cols_all].isna().mean().sort_values(ascending=False)

In [ ]:
def fetch_world_bank_country_metadata() -> pd.DataFrame:
    # per_page big enough for all entities
    url = "https://api.worldbank.org/v2/country?format=json&per_page=400"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    payload = r.json()
    records = payload[1] or []
    meta = pd.json_normalize(records)

    # Keep only what is needed
    keep = [
        "id",                      # ISO2
        "iso2Code",
        "name",
        "region.id",
        "region.value",
        "incomeLevel.id",
        "incomeLevel.value",
        "lendingType.id",
        "lendingType.value"
    ]
    meta = meta[keep].copy()
    meta.rename(columns={"name": "wb_name"}, inplace=True)
    return meta

meta = fetch_world_bank_country_metadata()
meta.head(), meta.shape

In [ ]:
# Join on country name
df2 = df.merge(meta, left_on="country", right_on="wb_name", how="left")

# Filter out aggregates: region.value == "Aggregates"
# Also remove rows where region.value is missing
before = df2.shape[0]
df2 = df2[df2["region.value"].notna()].copy()
df2 = df2[df2["region.value"] != "Aggregates"].copy()
after = df2.shape[0]

print("Rows before:", before)
print("Rows after removing aggregates:", after)

df2[["country","iso3","region.value","incomeLevel.value"]].head()

In [ ]:
# Baseline indicators for 2022 (WDI pull):
# - PM2.5 is fully missing in 2022 pull
# - homicide is very sparse (~54% missing)
INDICATORS_BASELINE = [
    "gdp_per_capita_ppp_const",
    "unemployment_rate",
    "labor_force_participation",
    "life_expectancy",
    "ppp_conversion_factor"
]

dfb = df2[["country","iso3","year"] + INDICATORS_BASELINE].copy()

missing_base = dfb[INDICATORS_BASELINE].isna().mean().sort_values(ascending=False)
missing_base

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(dfb[INDICATORS_BASELINE].isna(), cbar=False)
plt.title("Missingness heatmap (baseline indicators)")
plt.tight_layout()
plt.show()

In [ ]:
# Rule: drop countries missing more than 40% of baseline indicators
# (With 5 indicators, that means drop if missing >= 3 indicators)
max_missing_share = 0.40

row_missing_share = dfb[INDICATORS_BASELINE].isna().mean(axis=1)
dfb["missing_share"] = row_missing_share

before = dfb.shape[0]
dfb = dfb[dfb["missing_share"] <= max_missing_share].copy()
after = dfb.shape[0]

print("Countries before rule:", before)
print("Countries after rule:", after)
dfb["missing_share"].describe()

In [ ]:
# KNN imputation
imputer = KNNImputer(n_neighbors=5, weights="distance")

X = dfb[INDICATORS_BASELINE].to_numpy()
X_imp = imputer.fit_transform(X)

df_imp = dfb.copy()
df_imp[INDICATORS_BASELINE] = X_imp

# Confirm no missing values remain
df_imp[INDICATORS_BASELINE].isna().sum()

In [ ]:
OUT_PATH = "../data/processed/ali_2022_imputed.csv"
df_imp.drop(columns=["missing_share"]).to_csv(OUT_PATH, index=False)
OUT_PATH, df_imp.shape

In [ ]:
df_imp[INDICATORS_BASELINE].describe().T

In [ ]:
missing_base = df2[INDICATORS_BASELINE].isna().mean().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
missing_base.plot(kind="bar")
plt.title("Baseline indicator missingness (share missing)")
plt.ylabel("Share missing")
plt.tight_layout()

fig_path = "../outputs/figures/missingness_baseline.png"
plt.savefig(fig_path, dpi=200)
fig_path